## PhDAI733 Week 5 Group Project Part 1: Machine Learning Classification of Coronary Artery Disease

**Project Team:** Jude Adenuga, Pranathy Anumula, Rashidah Carr, Gino Varghese

**Deliverable:** Data Preprocessing and Initial Model Development

**Dataset:** Heart Disease, UCI Machine Learning Repository

**How this notebook is organized**

| Section |   
|---|
| Introduction and Project Framing |   
| 1. Setup and Imports |   
| 2. Uploading or Accessing Data |   
| 3. Data Exploration and Cleaning |   
| 4. Feature Engineering and Preprocessing |   
| 5. Train-Test Split |   
| 6. Model Training and Evaluation |   
| 7. Hyperparameter Tuning |   
| 8. Ethical Considerations |   
| 9. Team Collaboration Process |   
| 10. Next Steps |   





## Introduction and Project Framing

### Clinical problem

Coronary artery disease is the largest component of cardiovascular disease, the leading cause of death worldwide. Its definitive test, coronary angiography, is invasive and expensive, so clinicians decide who needs it from a first-line workup including history, symptoms, a resting ECG, blood work, and an exercise stress test. A model that estimates the probability of significant coronary narrowing from that workup could, once validated, help a clinic prioritize who goes to angiography first.

Part 1 builds and evaluates that initial model; ranking patients by risk   
Part 2 tasks could include setting a threshold at which a patient can safely be managed with follow-up care  (see Section 10).

### Stakeholders

- **Cardiologists and primary-care clinicians** (primary stakeholder): want a risk score that supports their referral decision and gives a clear reason behind each score.
- **Hospital quality and safety leadership**: need evidence that the model does not miss disease (false negatives are the costly error) and that it performs consistently across patient groups.
- **Patients**: bear the consequences of both a missed diagnosis and an unnecessary invasive procedure. They also deserve an explainable outcome.

### Analytics question

> Using only the information available after a patient history, resting ECG, basic blood tests, and a standard exercise test, can we predict whether a patient has angiographically significant coronary artery disease (more than 50 percent narrowing in a major vessel)?

This is a **binary classification** problem. The UCI target `num` records the number of major vessels with more than 50 percent narrowing (0 to 4); following standard practice for this dataset, it is collapsed to `disease` = 1 (any disease, `num` > 0) versus 0 (no disease). Section 3 explains why classification is preferred over regression and why the fluoroscopy (`ca`) and thallium (`thal`) results are held out of the model.

### Success criteria for Part 1

1. Audit and clean the dataset with a well documented and defensible decision for every issue found.
2. Build an initial supervised model from the first-line workup only, using a leakage-safe preprocessing pipeline.
3. Report the metrics a clinical stakeholder cares about:    
    - recall (sensitivity),
    - precision, and
    - F1 for the disease class.
4. Tune the initial model with cross-validation, and document early fairness observations.

##1. Setup and Imports
All libraries are imported and warnings are suppressed so that the outputs stay readable.

### The data
The UCI Heart Disease dataset was collected in 1988 from patients who were referred for coronary angiography. Four hospitals contributed data: Cleveland, Budapest, Zurich/Basel, and Long Beach VA. While the original records contain 76 attributes, most published studies utilize a processed subset of 14 attributes. The Cleveland file is the most commonly used because it is the most complete, encompassing data from 303 patients across 14 columns. It is important to note that the data comes from patients specifically undergoing angiography, which may not represent the general population. This distinction matters when evaluating whether the model can be effectively applied in other contexts.

### The columns worth knowing.
| Column | Type | What it means |
|---|---|---|
| age | Number | Years |
| sex | Binary | 1 = male, 0 = female |
| cp | Category | Chest pain: 1 typical angina, 2 atypical angina, 3 non-anginal pain, 4 asymptomatic |
| trestbps | Number | Resting blood pressure (mm Hg) |
| chol | Number | Serum cholesterol (mg/dl) |
| fbs | Binary | Fasting blood sugar above 120 mg/dl |
| restecg | Category | Resting ECG: 0 normal, 1 ST-T abnormality, 2 left ventricular hypertrophy |
| thalach | Number | Maximum heart rate reached on the exercise test |
| exang | Binary | Chest pain brought on by exercise |
| oldpeak | Number | ST depression on exercise relative to rest |
| slope | Category | Slope of the peak exercise ST segment: 1 up, 2 flat, 3 down |
| ca | Count | Major vessels (0 to 3) coloured by fluoroscopy |
| thal | Category | Thallium scan: 3 normal, 6 fixed defect, 7 reversible defect |
| num | Ranked | Angiographic diagnosis, 0 (none) to 4, what we predict |

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn ucimlrepo --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)



sns.set_theme(style='whitegrid')
pd.set_option('display.width', 140)
pd.set_option('display.max_columns', None)
RANDOM_STATE = 42

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('pandas', pd.__version__)
import sklearn; print('scikit-learn', sklearn.__version__)

## 2. Uploading or Accessing Data
This notebook fetches the dataset directly from the UCI Machine Learning Repository with the `ucimlrepo` package (dataset ID 45). Data is fetched at runtime and no data file is stored in the GitHub repository.   

`heart.data.original` returns the Cleveland file (303 patients, 14 columns) with `?` markers already converted to missing values, and `heart.variables` documents each column's role, type, and whether it has missing values.

In [ ]:
# UCI ID 45 is "Heart Disease"
heart = fetch_ucirepo(id=45)
df = heart.data.original
print('Loaded from UCI (ucimlrepo id=45)')
print()
print(heart.variables[['name', 'role', 'type', 'missing_values']])

In [ ]:
print('Shape:', df.shape)
print()
print(df.head())
print()
print('Columns:', list(df.columns))

## 3. Data Exploration and Cleaning

Before modeling, the dataset is audited on three data-quality dimensions (completeness, consistency, and conformity), the issues found are resolved with a documented decision for each.

The cleaned data is explored in three ways:
 - the relationship between two columns
 - the spread of a single column
 - the comparison of disease rates across categories

The section closes with the two decisions that shape the model: treating the problem as binary classification, and holding the two late-stage tests (`ca` and `thal`) out of the predictors.

### Completeness:

This analysis checks for missing values in two forms:

true nulls, which `ucimlrepo` has already converted from the `?` markers in the UCI file, and disguised nulls, meaning zeros recorded for blood pressure, cholesterol, or maximum heart rate that no living patient could have.

The second check matters because a placeholder zero would pass a standard null count and then distort every statistic computed on that column.

In [ ]:
# --- Completeness -----------------------------------------------------------
print('Total nulls:', df.isna().sum().sum())
print(df.isna().sum()[df.isna().sum() > 0].to_string())
print('Rows with any null: %d of %d' % (df.isna().any(axis=1).sum(), len(df)))
print()
# Placeholder zeros: a living patient cannot have 0 blood pressure, cholesterol, or heart rate.
for col in ['trestbps', 'chol', 'thalach']:
    print('%-8s: %d zeros | min %5.0f | max %5.0f' % (col, (df[col] == 0).sum(), df[col].min(), df[col].max()))


### Consistency:

This analysis checks for duplicate records. A patient entered twice would be counted twice in every rate and could land in both the training and test sets. This would allow the model to see its test answers during training.

**Duplicates:** The command `df.duplicated().sum()` returned 0, indicating that there are no duplicate entries in the dataset.

In [ ]:
print()
print('Exact duplicate rows:', df.duplicated().sum())
print()

### Conformity:


**Range Checks:** All coded columns utilize only their documented codes, and all continuous values fall within broad plausibility bounds. It’s important to note that while these range checks confirm the file aligns with its documentation, they do not guarantee that each individual value is accurate. For example, if a blood pressure reading is incorrectly typed as 140 instead of 130, it would still pass the check.

In [ ]:
# --- Conformity -------------------------------------------------------------
rules = {'sex': (0, 1), 'cp': (1, 4), 'fbs': (0, 1), 'restecg': (0, 2), 'exang': (0, 1),
         'slope': (1, 3), 'ca': (0, 3), 'thal': (3, 7), 'num': (0, 4),
         'age': (18, 100), 'trestbps': (80, 220), 'chol': (100, 600),
         'thalach': (60, 220), 'oldpeak': (0, 7)}
#Rules after reading the data
ranges = pd.DataFrame({
    'observed min': [df[c].min() for c in rules],
    'observed max': [df[c].max() for c in rules],
    'allowed min': [lo for lo, hi in rules.values()],
    'allowed max': [hi for lo, hi in rules.values()],
}, index=list(rules))
ranges['outside range'] = [int(((df[c] < lo) | (df[c] > hi)).sum()) for c, (lo, hi) in rules.items()]
print('Range checks (observed against allowed):')
print(ranges)
print()
print('Total values outside their allowed range:', int(ranges['outside range'].sum()))
print('Thallium scan:', sorted(df['thal'].dropna().astype(int).unique().tolist()))

---

**Assessment after performing data cleanup steps**

The dataset contains a small number of empty values.   

Six rows are missing data:
- four are missing values for 'ca' (coronary artery disease) and
- two for 'thal' (thalassemia).

No other fields in the dataset are empty.

Regarding placeholder zeros, there are none present. The minimum values for blood pressure, cholesterol, and heart rate are 94, 126, and 71 respectively  and are all within a plausible range. This is one reason the Cleveland dataset is considered the standard because the other three hospital datasets do not perform as well in this regard.   


**What We Did About It:**   
We removed the six incomplete rows, which represent 2.0% of the data. The main model does not use the 'ca' or 'thal' features so we could have kept these rows. However, we removed those so that the main model can be compared with one that includes those features. By dropping the incomplete rows before training, we ensured that both models were trained and tested on the same set of patients. Any differences observed will therefore result from the features rather than the patient data. The cost of removing these rows is minimal, and they are distributed across both outcomes, which are four patients without the disease and two with it.



In [ ]:
model_df = df.dropna().copy()
print('Rows dropped (missing ca or thal): %d of %d (%.1f%%)'
      % (len(df) - len(model_df), len(df), 100 * (len(df) - len(model_df)) / len(df)))
model_df['disease'] = (model_df['num'] > 0).astype(int)
print()
print(model_df['disease'].value_counts())
print()
print('Class balance:')
print(model_df['disease'].value_counts(normalize=True).round(3))

**Conclusion:** The data is clean. The key decision that shapes the model is not related to data quality but rather to which columns a real clinic would have available at the time a prediction is needed.

---

### Now we have to look at the data in three ways.

1) Relationship between two columns
2) Spread
3) Comparison across categories



**Relationship between two columns.** Figure 1a above plots maximum heart rate against age. Heart rate falls with age for everyone, but at any given age the patients with disease tend to sit lower, which means age and heart rate carry overlapping information.

In [ ]:
# Figure 1a. Max heart rate vs. age
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
sns.scatterplot(data=df, x='age', y='thalach', hue=(df['num'] > 0).map({True: 'Disease', False: 'No disease'}),
                alpha=0.6, s=20)
ax.set_title('Figure 1a. Max heart rate vs. age')
ax.set_ylabel('thalach (max heart rate)')
plt.tight_layout()
plt.show()

**Spread**: how values are distributed. Figure 2a is a histogram of age for the whole group. Figure 2b is a box-and-whisker plot of maximum heart rate split by outcome: the box covers the middle 50% of patients (25th to 75th percentile), the line inside is the median, and the whiskers reach toward the extremes. When comparing two groups, the box plot shows the difference in spread more directly than two overlapping histograms.

In [ ]:
# Figure 2a. Distribution of age (histogram)
# Figure 2b. Max heart rate by outcome (box and whiskers)


plot_df = model_df.assign(outcome=np.where(model_df['disease'] == 1, 'Disease', 'No disease'))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=plot_df, x='age', bins=20, ax=axes[0])
axes[0].set_title('Figure 2a. Distribution of age (histogram)')
sns.boxplot(data=plot_df, x='outcome', y='thalach', order=['No disease', 'Disease'], ax=axes[1])
axes[1].set_title('Figure 2b. Max heart rate by outcome (box and whiskers)')
axes[1].set_xlabel('')
axes[1].set_ylabel('thalach')
plt.tight_layout()
plt.show()

print('Quartiles of max heart rate by outcome:')
print(plot_df.groupby('outcome')['thalach'].describe()[['25%', '50%', '75%']].round(1))


**Comparison across categories**. Figure 2c shows disease rates for three predictors. The dashed line is the overall disease rate, so each column is read against the whole group, and each column is labeled with its patient count.

In [ ]:
# Figure 2c. Disease rate by chest pain type
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
overall = 100 * model_df['disease'].mean()
for ax, col, label in zip(axes, ['cp', 'sex', 'exang'],
                          ['chest pain type', 'sex (1 = male)', 'exercise-induced angina']):
    rate = 100 * model_df.groupby(col)['disease'].mean()
    counts = model_df.groupby(col).size()
    rate.index = rate.index.astype(int)
    rate.plot(kind='bar', ax=ax, rot=0, edgecolor='black')
    ax.axhline(overall, color='red', ls='--', lw=1)
    ax.set_title('Disease rate by %s' % label)
    ax.set_ylabel('Disease rate (%)')
    ax.set_xlabel(col)
    ax.set_ylim(0, 105)
    for i, v in enumerate(rate.values):
        ax.text(i, v + 2, 'n=%d' % counts.iloc[i], ha='center', fontsize=8)
axes[0].set_title('Figure 2c. Disease rate by chest pain type')
plt.tight_layout()
plt.show()

In [ ]:
print('Overall disease rate: %.1f%%' % overall)
print()
print(model_df.groupby('cp')['disease'].agg(['size', 'mean']).round(3))
print()
print(model_df.groupby('sex')['disease'].agg(['size', 'mean']).round(3))
print()
print(model_df.groupby('exang')['disease'].agg(['size', 'mean']).round(3))

### Assessment: what the charts show
The most striking comparison is the type of chest pain experienced by patients, which contrasts with common intuition. Patients who report typical angina (chest pain = 1) have a disease rate of approximately 30%, while those who are asymptomatic (chest pain = 4) have a disease rate of around 73%. This phenomenon can be attributed to the referral effect mentioned earlier: an asymptomatic patient typically undergoes cardiac catheterization only if prompted by an abnormal test result or another issue. The referrals take into account not only the patient's symptoms but also the reason for their consultation. Additionally, exercise-induced angina and sexual activity help clearly differentiate the groups. Men comprise about two-thirds of the patient population and exhibit a disease rate that is roughly twice that of women.

The box plot (Figure 2b) reveals something the age histogram cannot: patients with the disease achieve a lower peak heart rate (median of about 142 compared to 161), and the two boxes show minimal overlap in their middle sections.

Before the review, I predict that cp_4 (asymptomatic), sex, and oldpeak will show large positive coefficients (increasing the odds of disease), while thalach will be the strongest protective factor.

**Why choose classification over regression?** The variable "num" ranges from 0 to 4, but the first question a clinician needs to answer is whether the disease is present at all. The UCI documentation confirms this, stating that published research on this dataset focuses on predicting the presence or absence of the disease. By treating the values 1 to 4 as a single class, we can avoid the issue of fitting four severity levels with as few as 13 patients in the smallest group.

**Why should Ca and Thal be excluded?** The goal is to create a model that helps determine who requires further testing. Calcium (Ca) is derived from fluoroscopy, which is the same imaging method used for the initial diagnosis, while Thallium (Thal) requires a nuclear stress scan. By the time either of these tests are conducted, most of the diagnostic work has already been completed. The primary model relies solely on information gathered after taking a patient history, performing a resting ECG, conducting basic blood tests, and completing a standard exercise test. assesses the implications of this restriction.

**Class Balance**: After data cleaning, 46.1% of the patients in the dataset have the disease, which amounts to 137 out of 297 individuals. Unlike many medical datasets, this one is relatively balanced, meaning that a basic model that always predicts "no disease" would only achieve about 54% accuracy. In this context, accuracy will provide useful information; however, it will not indicate the specific types of mistakes the model makes.

##4. Feature Engineering and Preprocessing

The exploratory analysis identified associations between heart disease and several patient characteristics, particularly chest pain type, exercise-induced angina, and maximum heart rate. Building on these findings, we selected 11 predictors representing demographic characteristics, cardiovascular measurements, and exercise-related responses. The variables `ca` and `thal` were excluded in accordance with the earlier problem framing, while `num` was excluded because the binary `disease` variable serves as our prediction target.

To preserve the meaning of each variable, continuous predictors are standardized, categorical predictors are one-hot encoded, and binary indicators are retained in their original form. A scikit-learn preprocessing pipeline ensures that transformations are fitted only on the training data, preventing test-set leakage.

The resulting dataset contains 297 patient records, including 160 without recorded heart disease and 137 with recorded heart disease. A stratified train-test split will preserve this class distribution during model development and evaluation.

In [ ]:

# Numerical measurements will be standardized.
continuous_features = [
    'age',
    'trestbps',
    'chol',
    'thalach',
    'oldpeak'
]

# These numeric codes represent categories, not measurements.
categorical_features = [
    'cp',
    'restecg',
    'slope'
]

# Binary variables are already represented as 0 or 1.
binary_features = [
    'sex',
    'fbs',
    'exang'
]

# Combine the 11 selected predictors.
# Exclude ca, thal, and the original target num.
feature_columns = (
    continuous_features
    + categorical_features
    + binary_features
)

# Use Gino's cleaned dataset from Section 3.
X = model_df[feature_columns].copy()
y = model_df['disease'].copy()

# Confirm that the selected data contains no missing values.
assert X.isna().sum().sum() == 0
assert y.isna().sum() == 0

# Define the preprocessing steps.
# Scaling and encoding will be fitted after the train-test split.
preprocessor = ColumnTransformer(
    transformers=[
        (
            'continuous',
            StandardScaler(),
            continuous_features
        ),
        (
            'categorical',
            OneHotEncoder(
                handle_unknown='ignore',
                drop='first'
            ),
            categorical_features
        ),
        (
            'binary',
            'passthrough',
            binary_features
        )
    ],
    remainder='drop'
)

# Review the selected predictors and target.
print('Predictor matrix:', X.shape)
print('Target:', y.name)

print('\nContinuous predictors:', continuous_features)
print('Categorical predictors:', categorical_features)
print('Binary predictors:', binary_features)

print('\nClass distribution:')
print(y.value_counts().sort_index())


##5. Train-Test Split

The dataset was divided into training (80%) and testing (20%) subsets using stratified random sampling. Stratification preserves the relative proportion of patients with and without recorded heart disease in both subsets. A fixed random state of 42 ensures reproducibility.

The training set will be used to fit the preprocessing pipeline and develop the classification model. The test set will remain separate and will be used for final evaluation on previously unseen observations.

In [ ]:
# Reserve 20% of the data for testing.
# Stratify to preserve the distribution of both outcome groups.
# Fix the random state so the split is reproducible.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Check the size of each dataset.
print('Training features:', X_train.shape)
print('Testing features:', X_test.shape)

print('\nTraining targets:', y_train.shape)
print('Testing targets:', y_test.shape)

# Check that stratification preserved the class distribution.
print('\nTraining class distribution:')
print(y_train.value_counts().sort_index())

print('\nTesting class distribution:')
print(y_test.value_counts().sort_index())

# Compare the proportion of patients with recorded heart disease.
print('\nDisease prevalence:')
print(f'Original dataset: {y.mean():.2%}')
print(f'Training set: {y_train.mean():.2%}')
print(f'Testing set: {y_test.mean():.2%}')


##6. Model Training and Evaluation

A Random Forest classifier was selected as the initial model to examine whether the patient characteristics could distinguish between the two outcome groups. Random Forest combines multiple decision trees and can capture nonlinear relationships and interactions between predictors.

The classifier is integrated with the preprocessing pipeline developed in Section 4. The pipeline is fitted using only the training data, while the test set is reserved for evaluation. Model performance is assessed using accuracy, precision, recall, F1-score, and a confusion matrix. Particular attention is given to recall for patients with recorded heart disease, since false negatives represent cases in which the model fails to identify a patient with the recorded condition.

In [ ]:

# Section 6: Model Training and Evaluation
# Combine preprocessing and classification in one pipeline.
model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ))
    ]
)

# Fit the preprocessing steps and model using training data only.
model.fit(X_train, y_train)

# Predict the outcomes for patients in the test set.
y_pred = model.predict(X_test)

# Evaluate the model's classification performance.
print('Random Forest: Initial Model Evaluation')
print('-' * 40)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.2%}')

print('\nClassification Report:')
print(classification_report(
    y_test,
    y_pred,
    target_names=[
        'No Heart Disease',
        'Heart Disease'
    ],
    digits=3,
    zero_division=0
))

# Visualize the correct and incorrect predictions.
cm = confusion_matrix(y_test, y_pred)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        'No Disease',
        'Disease'
    ]
).plot(cmap='Blues', values_format='d')

plt.title('Random Forest: Confusion Matrix')
plt.xlabel('Predicted Outcome')
plt.ylabel('Actual Outcome')
plt.tight_layout()
plt.show()

# Examine false negatives and false positives.
tn, fp, fn, tp = cm.ravel()

print('\nConfusion Matrix Summary:')
print('True negatives:', tn)
print('False positives:', fp)
print('False negatives:', fn)
print('True positives:', tp)


The initial Random Forest model achieved 80.0% accuracy on the 60-patient test set. It correctly identified 21 of the 28 patients with recorded heart disease, yielding a recall of 75.0% and an F1-score of 0.778 for the disease class. The confusion matrix identified seven false negatives and five false positives. Although these results demonstrate predictive performance on the held-out sample, the missed disease cases highlight the importance of examining recall alongside overall accuracy. The findings are preliminary and do not establish clinical reliability.

##7. Hyperparameter Tuning

Hyperparameter tuning was conducted using GridSearchCV with five-fold stratified cross-validation on the training set. The search evaluated different combinations of the number of trees, maximum tree depth, and minimum samples required for node splitting and leaf formation.

Disease recall was selected as the primary optimization metric because the initial model missed seven patients with recorded heart disease. The selected configuration was subsequently evaluated against the initial model using accuracy, precision, recall, F1-score, and the confusion matrix. As the initial test results had already been examined, this comparison is treated as exploratory.

In [ ]:

# Section 7: Hyperparameter Tuning

# Define the hyperparameters to investigate.
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [3, 5, None],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2]
}

# Use five-fold cross-validation with a reproducible split.
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Tune the complete pipeline using training data only.
# Prioritize recall to reduce missed disease cases.
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='recall',
    cv=cv,
    n_jobs=-1,
    refit=True
)

grid_search.fit(X_train, y_train)

# Review the selected hyperparameters and validation result.
print('Best Hyperparameters:')
print(grid_search.best_params_)

print('\nBest Cross-Validation Recall:')
print(f'{grid_search.best_score_:.2%}')

# Retrieve the tuned model and predict the test outcomes.
tuned_model = grid_search.best_estimator_
y_pred_tuned = tuned_model.predict(X_test)

# Evaluate the tuned model.
print('\nRandom Forest: Tuned Model Evaluation')
print('-' * 40)

print(f'Accuracy: {accuracy_score(y_test, y_pred_tuned):.2%}')

print('\nClassification Report:')
print(classification_report(
    y_test,
    y_pred_tuned,
    target_names=[
        'No Heart Disease',
        'Heart Disease'
    ],
    digits=3,
    zero_division=0
))

# Display the tuned model's confusion matrix.
cm_tuned = confusion_matrix(y_test, y_pred_tuned)

ConfusionMatrixDisplay(
    confusion_matrix=cm_tuned,
    display_labels=[
        'No Disease',
        'Disease'
    ]
).plot(cmap='Blues', values_format='d')

plt.title('Tuned Random Forest: Confusion Matrix')
plt.xlabel('Predicted Outcome')
plt.ylabel('Actual Outcome')
plt.tight_layout()
plt.show()

# Compare the initial and tuned model performance.
comparison = pd.DataFrame({
    'Metric': [
        'Accuracy',
        'Disease Precision',
        'Disease Recall',
        'Disease F1-score'
    ],
    'Initial Model': [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred)
    ],
    'Tuned Model': [
        accuracy_score(y_test, y_pred_tuned),
        precision_score(y_test, y_pred_tuned),
        recall_score(y_test, y_pred_tuned),
        f1_score(y_test, y_pred_tuned)
    ]
})

print('\nInitial vs Tuned Model:')
print(comparison.round(3).to_string(index=False))

# Compare the number of missed disease cases.
print('\nInitial false negatives:',
      confusion_matrix(y_test, y_pred)[1, 0])

print('Tuned false negatives:', cm_tuned[1, 0])


Hyperparameter tuning did not improve the model on the held-out test set. The configuration selected by cross-validation (200 trees, unlimited depth, at least five samples per split and two per leaf) reached a cross-validated disease recall of 74.3%, but on the 60-patient test set its accuracy fell from 80.0% to 76.7% and its disease recall from 75.0% to 71.4%, with false negatives rising from seven to eight. With a test set this small, one patient is 1.7 percentage points of accuracy and 3.6 points of recall, so the two configurations are statistically indistinguishable. A fair assessment is that tuning on recall alone neither helped nor hurt. These findings remain preliminary and do not establish clinical reliability.

## 8. Ethical Considerations
- What potential biases exist in the dataset?
- How would the model perform across different demographic groups?
- Is the model interpretable and fair?

Each question is answered below with evidence from this notebook.

### 8.1 What potential biases exist in the dataset? Could the model reflect or reinforce them?

Three biases are visible in the data itself, before any model is trained.

**Sex imbalance.** The sample is two-thirds male (201 men, 96 women), and disease prevalence is 56 percent in men against 26 percent in women (Section 3). The model therefore learns from about 112 diseased men but only 25 diseased women. With so few examples, it has less opportunity to learn how disease presents in women, and clinical research has long documented that coronary disease is under-diagnosed in women because their symptoms differ from the male pattern. A model trained on this sample could carry that under-diagnosis forward.

**Referral bias.** Every patient in the file was referred for angiography. In this case the sample is not the general population, it is the subset of patients whose clinicians already suspected disease. Section 3 showed the clearest symptom of this, that asymptomatic patients have the highest disease rate (73 percent), because an asymptomatic patient reaches angiography only when another test was abnormal. The model learns that referral pattern as if it were physiology, and would score a walk-in patient with no symptoms very differently from how a clinician would.

**Single site and era.** All 297 patients were seen at one hospital (the Cleveland Clinic) in the 1980s. Diagnostic practice, patient demographics, and the tests that were routinely ordered have all changed since. A model that fits Cleveland well may not transfer to another hospital or to today's patients.

The first two biases are discussed in this notebook in Section 8.2. The third can only be tested against external data, which is one of the potential next steps.

### 8.2 How would the model perform across different demographic groups?

The notebook evaluates the model on the test set as a whole, instead of breaking the results down by sex or age.

**By sex.** Section 3 showed that men in this sample have twice the disease rate of women (56 percent against 26 percent), and Section 4 kept `sex` as a predictor. The model has therefore learned that being male raises the probability of disease, which is true on average in this sample but works against the individual woman who does have disease because her score starts lower and she needs stronger findings on the other predictors to cross the decision threshold. With only 25 diseased women in the whole dataset and after the 80/20 split, roughly 5 in the test set, the model has had few examples of what disease looks like in women. The seven false negatives in Section 6 are likely to include a disproportionate share of them. The reverse risk also exists in that a healthy man with asymptomatic chest pain may be scored high on those two features alone. Which effect dominates cannot be determined without group-level metrics.

**By age.** Age is one of the eleven predictors, and disease prevalence rises with age in this sample, so the model has learned to treat youth as protective. A patient under 45 who does have disease looks low-risk on age and often on maximum heart rate too (Section 3, Figure 1a). Such patients are the ones most likely to be missed, and the same 60-patient test set contains too few of them to detect it.

**What this means.** The overall recall of 75 percent (Section 6) is an average across groups, and averages hide the groups that are missed most. The seven missed patients are the place to look: the demographic mix of those seven, compared with the 21 who were caught, would show whether the errors fall evenly or concentrate among women and younger patients. Computing sensitivity and the false-negative rate separately for each sex and age band, on both the test set and cross-validated predictions from the larger training set, is the first fairness task for Part 2, and it is the evidence required before this model could be described as performing consistently across patient groups.


### 8.3 Is the model interpretable and fair? How transparent is its decision-making?

A Random Forest is not directly interpretable in the way a logistic regression is, in that no single coefficient tells a clinician why a patient scored high. It can still be made transparent enough though for decision support. Feature importances from the tuned pipeline show which inputs drive the score, and they should line up with the exploratory findings in Section 3.   
The model is proposed as decision support only as it ranks patients for angiography priority and never makes the referral decision itself. On fairness, the group gaps mean the model is not yet fair enough to deploy without mitigations.


##9. Team Collaboration Process

**Communication**   
Our team used a collaborative and organized approach throughout the project so that both the technical analysis and this written report reflect decisions the whole group made together. At the beginning of the project we discussed the overall objective, selected the Heart Disease dataset, and agreed that the main goal would be to predict the presence of coronary artery disease using only the information available during the initial clinical workup. Responsibilities were then divided among team members according to the major stages of the analytics process, which meant that one member owned data exploration and cleaning, another owned preprocessing and the train-test split, another owned model development, evaluation, and tuning, and the remaining member owned the problem framing, the ethical analysis, and the assembly of this report, with each stage of the notebook building directly on the one before it.  


**Tools**   
Google Colab served as the primary coding environment because it allowed every team member to open the same Python notebook, test code, and compare results without maintaining separate local configurations, and because the notebook fetches the dataset directly from UCI at run time, everyone worked from identical data. GitHub was used for version control and to maintain an organized record of changes to the project files, with the notebook committed with outputs cleared so that differences between versions stay readable and a decisions log recording the choices that later sections depend on. Outlook email and Blackboard messages were the team’s main communication channels, and we used them to share project updates, discuss task responsibilities, exchange files and links, communicate deadlines, and follow up on decisions that required input from more than one person. Keeping those exchanges in writing meant that important information was documented and available to everyone, which mattered most when team members were working at different times.   


**Decisions**   
Team members communicated regularly to review findings, identify issues, and agree on methodological decisions before they were included in the final analysis. The clearest example is the choice of clinical variables, where the team discussed which predictors belonged in the model and agreed to exclude the fluoroscopy vessel count and the thallium scan because both are obtained late in the diagnostic process, after the point at which a referral model would be used. We also reviewed the importance of preventing data leakage by fitting all preprocessing inside the pipeline on training data only, selected evaluation metrics suited to a clinical setting, and agreed to emphasize recall because a false negative is a patient with disease whom the model would fail to refer. After evaluating the initial and tuned Random Forest models, the team compared the results and discussed the limitations that a small dataset, uneven demographic representation, and possible referral bias place on what the model can claim. Working through those decisions together in Colab, GitHub, and written communication allowed the team to coordinate responsibilities, keep everyone informed, and make each technical decision collaboratively from the start of the project to the end.



##10. Potential Next Steps

1. **Add a baseline and a second model.** Compare the Random Forest against a majority-class baseline and a logistic regression, and report ROC-AUC alongside recall so the models can be compared independently of the decision threshold.
2. **Tune the decision threshold.** Section 7 showed that adjusting tree parameters did not raise recall. We can evaluate lowering the probability threshold from the default 0.5 for the groups that are found to be missed most often (patients 45 and under, and men with a low-risk symptom profile).
3. **Test the model on the other three hospitals.** The Hungarian, Swiss, and VA files in the same UCI dataset offer a ready-made check on whether a Cleveland-trained model holds up elsewhere.
